# Crypto Sentiment Analysis

Exploratory notebook for analyzing the relationship between Bitcoin Fear/Greed sentiment and Hyperliquid trader performance.

Expected raw files:

- `../data/raw/fear_greed_index.csv`
- `../data/raw/historical_data.csv`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.analysis import run_advanced_analytics, run_exploratory_analysis
from src.data_loader import load_project_datasets
from src.feature_engineering import (
    engineer_features,
    merge_daily_performance_with_sentiment,
    merge_trades_with_sentiment,
)
from src.preprocessing import preprocess_datasets
from src.utils import RAW_DATA_DIR, CHARTS_DIR, REPORTS_DIR
from src.visualization import generate_visualizations

PROJECT_ROOT

## Load and Prepare Data

In [ ]:
required_files = [RAW_DATA_DIR / "fear_greed_index.csv", RAW_DATA_DIR / "historical_data.csv"]
missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    print("Raw data files are missing:")
    for path in missing_files:
        print(f"- {path.relative_to(PROJECT_ROOT)}")
else:
    fear_greed, trader_history = load_project_datasets()
    cleaned_fear_greed, cleaned_trader_history, reports = preprocess_datasets(
        fear_greed.dataframe,
        trader_history.dataframe,
    )
    sentiment_features, trade_features, trader_metrics = engineer_features(
        cleaned_fear_greed,
        cleaned_trader_history,
    )
    trade_sentiment = merge_trades_with_sentiment(trade_features, sentiment_features)
    daily_sentiment = merge_daily_performance_with_sentiment(trade_features, sentiment_features)
    print("Data loaded and prepared.")

## Dataset Snapshots

In [ ]:
if not missing_files:
    display(sentiment_features.head())
    display(trade_features.head())
    display(trader_metrics.head())

## Exploratory Analysis

In [ ]:
if not missing_files:
    analysis_results = run_exploratory_analysis(
        trade_sentiment,
        daily_sentiment,
        trader_metrics,
    )
    display(analysis_results.sentiment_profitability)
    display(analysis_results.symbol_performance.head(10))

## Advanced Trader Analytics

In [ ]:
if not missing_files:
    advanced_results = run_advanced_analytics(trade_sentiment, trader_metrics)
    display(advanced_results.top_traders)
    display(advanced_results.worst_traders)
    display(advanced_results.statistical_tests)

## Visualizations

In [ ]:
if not missing_files:
    chart_paths = generate_visualizations(
        trade_sentiment,
        daily_sentiment,
        trader_metrics,
        analysis_results,
        CHARTS_DIR,
    )
    chart_paths